In [1]:
from itertools import pairwise
import ray
import torch
import torch.nn.functional as F
import numpy as np

In [3]:
# Define the square task.
@ray.remote(num_cpus=0.75)
def square(x):
    N = 1000
    reps = 10
    eps = 1e-6
    arr = np.random.randn(N, N)
    res = np.zeros_like(arr)
    for _ in range(reps):
        norm = np.linalg.norm(res) + eps
        res += (arr @ arr) / norm
    return np.sum(arr @ arr)

In [4]:
# Interesting helper functions

# ray.init(address="auto")

GPU_FRACTION=0.1
CPU_FRACTION=0#0.05

# @ray.remote(num_gpus=GPU_FRACTION)
def rand_matmul(n, r, device):
    # print(f"rand_mat: {n=}, {m=}, {r=}")
    A = torch.randn(n, r).to(device)
    B = torch.randn(n, r).to(device)
    # print(f"rand_matmul: [{list(A.shape)}] @ [{list(B.shape)=}]")
    return torch.einsum("ar,br->ab", A, B)

# @ray.remote(num_gpus=GPU_FRACTION*4)
def bridge_tensor(a, c, device):
    """returns a matrix b of the smallest shape so that: a @ b @ c is valid"""
    # print(f"bridge_tensor: {a=}, {c=}")
    return torch.randn(a.shape[-1], c.shape[0]).to(device)

@ray.remote(num_gpus=GPU_FRACTION, num_cpus=CPU_FRACTION)
def mul_chain(n=10_000, r=1_000, l=2) -> torch.Tensor | None:
    # print(f"mul_chain: {n=}, {r=}, {l=}")
    device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
    shape = [n, n]

    res = torch.ones(shape).to(device)
    for _ in range(l):
        a, c = [rand_matmul(n=n, r=r, device=device) for _ in range(2)]
        b = bridge_tensor(a, c, device=device)
        for _ in range(2**8): # This inner loop produces high arithmetic intensity
            res = F.rms_norm(res + a @ b @ c, normalized_shape=shape)
        # print(f"mul_chain: {list(d.shape)} = {list(a.shape)} @ {list(b.shape)} @ {list(c.shape)}")
    return torch.einsum("ab,bc->", res, res).to("cpu")

In [5]:
# Make the GPUs sweat
NUM_GPU = 4
num_tasks = min(int(NUM_GPU / GPU_FRACTION), 1_000)
print(f"{num_tasks=}")
mc_ftrs = [mul_chain.remote(n=2**10, r=2**10, l=128) for _ in range(num_tasks)]
sq_ftrs = [square.remote(i) for i in range(10_000)]

2026-05-29 15:04:44,159	INFO worker.py:1814 -- Connecting to existing Ray cluster at address: 10.0.1.14:6379...
2026-05-29 15:04:44,191	INFO worker.py:2003 -- Connected to Ray cluster. View the dashboard at http://127.0.0.1:8265 


num_tasks=40


/home/nico/miniconda3/envs/drp/lib/python3.14/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


In [5]:
sqs_and_mcs = ray.get(sq_ftrs + mc_ftrs)
 # = ray.get(mc_ftrs)
# [mc for mc in mcs]

In [3]:
# # Launch four parallel square tasks.
# futures = [square.remote(i) for i in range(10_000)]

# futures[:10]

In [4]:
# # Retrieve results.
# print(ray.get(futures)[:5])

In [2]:
@ray.remote
class ActorTest:
    def __init__(self):
        self.val = 0

    def inc(self):
        self.val += 1
        return self.val

    def get_val(self):
        return self.val


In [3]:
a = ActorTest.remote()

2026-05-29 16:24:21,481	INFO worker.py:1814 -- Connecting to existing Ray cluster at address: 10.0.1.14:6379...
2026-05-29 16:24:21,512	INFO worker.py:2003 -- Connected to Ray cluster. View the dashboard at http://127.0.0.1:8265 
/home/nico/miniconda3/envs/drp/lib/python3.14/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


In [24]:
val_ref = a.inc.remote()
ray.get(val_ref)

20

1

In [ ]:
# (0, 1, 2)
a = torch.randn(2, 3, 4)
print(a)

In [ ]:
a[0, 0, 2]

In [ ]:
ap = a.permute(2, 0, 1)
print(ap.shape)
print(ap)

In [ ]:
futures = [matmul_gpu.remote() for i in range(1000)]

results = ray.get(futures)
for r in results:
    print(r)

In [ ]:
# from ray.train import ScalingConfig
# from ray.train.torch import train_loop_utils, TorchTrainer
# from torchvision.datasets import CIFAR10
# from torchvision.transforms import ToTensor

# scaling_config = ScalingConfig(
#     num_workers=2,
#     use_gpu=True
# )

In [ ]:
# # 1. Download/load the dataset using torchvision
# torch_dataset = CIFAR10(root="/home/nico/ray-test/notebooks/data", train=True, download=True, transform=ToTensor())

# # 2. Convert to Ray Data
# ray_dataset = ray.data.from_torch(torch_dataset)

# print(ray_dataset.count())

In [3]:
import ray
import time

# A regular Python function.
def normal_function():
    return 1

# By adding the `@ray.remote` decorator, a regular Python function
# becomes a Ray remote function.
@ray.remote
def my_function():
    return 1

# To invoke this remote function, use the `remote` method.
# This will immediately return an object ref (a future) and then create
# a task that will be executed on a worker process.
obj_ref = my_function.remote()

# The result can be retrieved with ``ray.get``.
assert ray.get(obj_ref) == 1
ray.get(obj_ref)

2026-05-29 14:15:52,664	INFO worker.py:1814 -- Connecting to existing Ray cluster at address: 10.0.1.14:6379...
2026-05-29 14:15:52,697	INFO worker.py:2003 -- Connected to Ray cluster. View the dashboard at http://127.0.0.1:8265 
/home/nico/miniconda3/envs/drp/lib/python3.14/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


1